# Motor Imagery Decoder — Train OFFLINE, Evaluate ONLINE (FES vs NOFES)

For each subject × offline-session, train a CSP + LDA classifier on the OFFLINE recording, then apply it to the two matched ONLINE sessions.

**Pair 1:** train on `S001 OFFLINE_FES` → test on `S002 ONLINE_FES` and `S003 ONLINE_NOFES`
**Pair 2:** train on `S004 OFFLINE_NOFES` → test on `S006 ONLINE_FES` and `S005 ONLINE_NOFES`

**Metrics reported per (subject × pair × condition):**
1. **Classification accuracy** — fraction of cued trials correctly classified
2. **Classification amplitude** — mean |LDA decision-function value|
3. **SNR** — (a) Fisher ratio of the LDA projection on online data, and (b) mu-band power ratio REST / MI over motor channels C3/Cz/C4

In [13]:
# Install dependencies if needed
# !pip install pyxdf mne scipy numpy matplotlib

In [ ]:
import os
import re
import glob
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import pyxdf
from scipy.signal import welch, butter, filtfilt
from scipy.linalg import eigh

plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})

## Configuration

In [ ]:
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'Group 2 - Glove')

# Marker codes (same cue-encoding in offline and online)
MI_BEGIN, MI_END     = 200, 220
REST_BEGIN, REST_END = 100, 120
TARGET_MARKERS       = [100, 120, 200, 220]

# Epoch window (t=0 at cue marker)
T_PRE  = -1.0   # baseline start
T_POST =  5.0   # epoch end

# Bandpass for MI decoding (mu + beta)
BP_LO, BP_HI = 8.0, 30.0

# CSP spatial filters (top N/2 + bottom N/2)
N_CSP = 4

NON_EEG = {'AUX1', 'AUX2', 'AUX3', 'AUX7', 'AUX8', 'AUX9', 'TRIGGER'}
RENAME  = {'FP1':'Fp1','FPZ':'Fpz','FP2':'Fp2','FZ':'Fz','CZ':'Cz',
           'PZ':'Pz','POZ':'POz','OZ':'Oz'}

MOTOR_CH = ['C3', 'Cz', 'C4']
MU_BAND  = (8, 13)

# Design: per subject, each OFFLINE session trains a model tested on two ONLINE sessions
PAIRS = [
    {'name':  'Pair1 (train=OFFLINE_FES)',
     'train': 'S001', 'online_fes': 'S002', 'online_nofes': 'S003'},
    {'name':  'Pair2 (train=OFFLINE_NOFES)',
     'train': 'S004', 'online_fes': 'S006', 'online_nofes': 'S005'},
]

## Helper Functions

In [ ]:
# ── XDF loading + session parsing ─────────────────────────────────────────────

def get_channel_names_from_xdf(eeg_stream):
    ch_desc  = eeg_stream['info']['desc'][0]
    channels = ch_desc.get('channels', [{}])[0].get('channel', [])
    return [ch['label'][0] for ch in channels]


# Tolerates the 'ONOLINE' typo in subj 003 / S005
_SESSION_RE = re.compile(r'ses-(S\d+)(O[A-Z]*LINE)_(FES|NOFES)')
_SUBJ_RE    = re.compile(r'SUBJ_(\d+)')

def parse_session(path):
    """Return (subject, session_id, kind, stim) or None."""
    base    = os.path.basename(path)
    m_subj  = _SUBJ_RE.search(base)
    m_ses   = _SESSION_RE.search(base)
    if not (m_subj and m_ses):
        return None
    ses_id, raw_kind, stim = m_ses.group(1), m_ses.group(2), m_ses.group(3)
    kind = 'OFFLINE' if 'OFF' in raw_kind else 'ONLINE'
    return m_subj.group(1), ses_id, kind, stim


def load_xdf_file(filepath):
    streams, _ = pyxdf.load_xdf(filepath)

    eeg_stream = marker_stream = None
    for s in streams:
        stype = s['info']['type'][0].lower()
        if stype == 'eeg':       eeg_stream    = s
        elif stype == 'markers': marker_stream = s
    if eeg_stream is None or marker_stream is None:
        eeg_stream    = streams[0]
        marker_stream = streams[1] if len(streams) > 1 else None

    eeg_timestamps = np.array(eeg_stream['time_stamps'])
    eeg_data       = np.array(eeg_stream['time_series']).T
    channel_names  = get_channel_names_from_xdf(eeg_stream)
    sfreq          = float(eeg_stream['info']['nominal_srate'][0])

    valid_idx     = [i for i, ch in enumerate(channel_names) if ch not in NON_EEG]
    channel_names = [channel_names[i] for i in valid_idx]
    eeg_data      = eeg_data[valid_idx, :]
    channel_names = [RENAME.get(ch, ch) for ch in channel_names]

    ts_arr      = np.asarray(marker_stream['time_series'], dtype=float)
    marker_data = ts_arr[:, 0].astype(int)
    marker_ts   = ts_arr[:, 1]
    keep        = np.isin(marker_data, TARGET_MARKERS)
    return eeg_data, eeg_timestamps, marker_data[keep], marker_ts[keep], channel_names, sfreq


def bandpass(data, lo, hi, sfreq, order=4):
    nyq  = sfreq / 2.0
    b, a = butter(order, [max(lo, 0.5) / nyq, min(hi, nyq - 0.1) / nyq], btype='band')
    return filtfilt(b, a, data, axis=-1)


def extract_epochs(eeg_data, eeg_ts, marker_data, marker_ts, sfreq, begin_code,
                   t_pre=T_PRE, t_post=T_POST):
    """Returns (n_epochs, n_ch, n_samp) — all epochs trimmed to the same length."""
    epochs = []
    n_pre  = int(abs(t_pre) * sfreq)

    for bi in np.where(marker_data == begin_code)[0]:
        t_start = marker_ts[bi]
        i0 = np.searchsorted(eeg_ts, t_start + t_pre)
        i1 = np.searchsorted(eeg_ts, t_start + t_post)
        if i0 < 0 or i1 > eeg_data.shape[1]:
            continue
        ep = eeg_data[:, i0:i1].copy()
        if ep.shape[1] > n_pre:
            ep -= ep[:, :n_pre].mean(axis=1, keepdims=True)
        epochs.append(ep)

    if not epochs:
        return np.empty((0, eeg_data.shape[0], 0))
    min_len = min(e.shape[-1] for e in epochs)
    return np.stack([e[:, :min_len] for e in epochs])


def load_session_epochs(filepath):
    """Bandpass continuous EEG, epoch on MI/REST cues, keep ACTIVE window [0, T_POST].
    Returns X (n_trials, n_ch, n_samp), y (1=MI, 0=REST), ch_names, sfreq.
    """
    eeg, eeg_ts, mk, mk_ts, ch_names, sfreq = load_xdf_file(filepath)
    eeg_bp = bandpass(eeg, BP_LO, BP_HI, sfreq)

    mi   = extract_epochs(eeg_bp, eeg_ts, mk, mk_ts, sfreq, MI_BEGIN)
    rest = extract_epochs(eeg_bp, eeg_ts, mk, mk_ts, sfreq, REST_BEGIN)

    n_pre = int(abs(T_PRE) * sfreq)
    if mi.shape[-1]   > n_pre: mi   = mi[..., n_pre:]
    if rest.shape[-1] > n_pre: rest = rest[..., n_pre:]

    n = min(mi.shape[-1], rest.shape[-1]) if (mi.size and rest.size) else 0
    mi, rest = mi[..., :n], rest[..., :n]

    X = np.concatenate([mi, rest], axis=0)
    y = np.concatenate([np.ones(len(mi), int), np.zeros(len(rest), int)])
    return X, y, ch_names, sfreq


# ── CSP + LDA (2-class, numpy/scipy only) ────────────────────────────────────

def _mean_cov(X):
    """Average trace-normalized trial covariance. X: (n_trials, n_ch, n_samp)."""
    covs = np.einsum('ijk,ilk->ijl', X, X)
    covs /= np.trace(covs, axis1=1, axis2=2)[:, None, None]
    return covs.mean(axis=0)


class CSPLDA:
    """Common Spatial Patterns (log-var features) + Linear Discriminant Analysis."""

    def __init__(self, n_csp=N_CSP, reg=1e-6):
        self.n_csp = n_csp
        self.reg   = reg

    def fit(self, X, y):
        assert set(np.unique(y)) == {0, 1}, 'CSPLDA requires both classes in training set'
        C1 = _mean_cov(X[y == 1])
        C0 = _mean_cov(X[y == 0])
        # Generalized eigenproblem: C1 w = λ (C0 + C1) w
        evals, evecs = eigh(C1, C0 + C1)
        order  = np.argsort(evals)
        k      = self.n_csp // 2
        # Stack bottom-k (max class-0 variance) and top-k (max class-1 variance) filters
        self.filters_ = np.concatenate([evecs[:, order[:k]],
                                        evecs[:, order[-k:]]], axis=1).T    # (n_csp, n_ch)

        F = self._features(X)
        mu1, mu0 = F[y == 1].mean(0), F[y == 0].mean(0)
        Sw = np.cov(F[y == 1].T, ddof=1) + np.cov(F[y == 0].T, ddof=1)
        Sw += self.reg * np.eye(Sw.shape[0])
        self.coef_      = np.linalg.solve(Sw, mu1 - mu0)
        self.intercept_ = -self.coef_ @ ((mu1 + mu0) / 2)
        return self

    def _features(self, X):
        Z   = np.einsum('fc,ncs->nfs', self.filters_, X)          # spatial filter
        var = Z.var(axis=-1, ddof=1)
        return np.log(var / var.sum(axis=1, keepdims=True))       # normalized log-var

    def decision_function(self, X):
        return self._features(X) @ self.coef_ + self.intercept_

    def predict(self, X):
        return (self.decision_function(X) > 0).astype(int)


# ── Evaluation metrics ───────────────────────────────────────────────────────

def evaluate(clf, X, y):
    margin = clf.decision_function(X)
    pred   = (margin > 0).astype(int)
    acc    = (pred == y).mean()
    amp    = np.abs(margin).mean()
    m1, m0 = margin[y == 1], margin[y == 0]
    fisher = (m1.mean() - m0.mean()) ** 2 / (m1.var(ddof=1) + m0.var(ddof=1) + 1e-30)
    return dict(acc=acc, amp=amp, fisher=fisher, margin=margin, y=y, pred=pred)


def spectral_snr(X, y, ch_idx, sfreq, band=MU_BAND):
    """Ratio of REST mu-power to MI mu-power averaged over selected channels."""
    def band_pwr(sig):
        f, p = welch(sig, fs=sfreq,
                     nperseg=min(int(sfreq * 2), sig.shape[-1]),
                     noverlap=int(sfreq), axis=-1)
        m = (f >= band[0]) & (f < band[1])
        return np.trapezoid(p[..., m], f[m], axis=-1).mean()

    return band_pwr(X[y == 0][:, ch_idx, :]) / (band_pwr(X[y == 1][:, ch_idx, :]) + 1e-30)

## Load Data

In [ ]:
xdf_files = sorted(glob.glob(os.path.join(DATA_DIR, '*.xdf')))
print(f'Found {len(xdf_files)} XDF file(s).\n')

# sessions[subject][session_id] = dict(X, y, kind, stim, ch_names, sfreq, file)
sessions = {}

for fp in xdf_files:
    meta = parse_session(fp)
    if meta is None:
        print(f'  SKIP (unparsed): {os.path.basename(fp)}')
        continue
    subj, ses_id, kind, stim = meta
    try:
        X, y, ch_names, sfreq = load_session_epochs(fp)
    except Exception as e:
        print(f'  ERROR {os.path.basename(fp)}: {e}')
        continue

    sessions.setdefault(subj, {})[ses_id] = dict(
        X=X, y=y, kind=kind, stim=stim,
        ch_names=ch_names, sfreq=sfreq, file=os.path.basename(fp))

    print(f'  {subj}/{ses_id}  {kind:<7} {stim:<5}  '
          f'n={len(y):3d}  (MI={int(y.sum())}, REST={int((1-y).sum())})')

subjects = sorted(sessions.keys())
print(f'\nLoaded {len(subjects)} subject(s): {subjects}')

## Verify Session Layout

In [ ]:
# Verify channel layout is consistent across sessions, locate motor channels
ref_subj = subjects[0]
ref_ses  = next(iter(sessions[ref_subj].values()))
channel_names_global = ref_ses['ch_names']
sfreq_global         = ref_ses['sfreq']

mismatches = [f'{subj}/{sid}' for subj in subjects for sid, s in sessions[subj].items()
              if s['ch_names'] != channel_names_global]
if mismatches:
    print('!! channel mismatch in:', mismatches)

motor_idx_global = [channel_names_global.index(c) for c in MOTOR_CH
                    if c in channel_names_global]

print(f'Channels ({len(channel_names_global)}): {channel_names_global}')
print(f'Sampling rate: {sfreq_global} Hz')
print(f'Motor channels {MOTOR_CH} → indices {motor_idx_global}')

## Train CSP + LDA on OFFLINE, Evaluate on ONLINE

In [ ]:
results = []   # one row per (subject, pair, condition)

for subj in subjects:
    subj_ses = sessions[subj]

    for pair in PAIRS:
        needed  = (pair['train'], pair['online_fes'], pair['online_nofes'])
        missing = [k for k in needed if k not in subj_ses]
        if missing:
            print(f'[{subj}] {pair["name"]}: missing {missing} — skipping')
            continue

        train = subj_ses[pair['train']]
        clf   = CSPLDA(n_csp=N_CSP).fit(train['X'], train['y'])
        train_acc = (clf.predict(train['X']) == train['y']).mean()

        for cond_key, cond_label in [('online_fes', 'FES'), ('online_nofes', 'NOFES')]:
            te    = subj_ses[pair[cond_key]]
            res   = evaluate(clf, te['X'], te['y'])
            snr_s = spectral_snr(te['X'], te['y'], motor_idx_global, te['sfreq'])

            results.append(dict(
                subject=subj, pair=pair['name'], condition=cond_label,
                train_file=train['file'], test_file=te['file'],
                train_acc=train_acc, n_test=len(te['y']),
                acc=res['acc'], amp=res['amp'], fisher=res['fisher'], mu_snr=snr_s,
                margin=res['margin'], y_test=res['y'], pred=res['pred'],
            ))

# Results table
hdr = f'{"Subj":<5} {"Pair":<28} {"Cond":<6} {"n":>4} {"trainAcc":>9} {"acc":>7} {"|marg|":>8} {"Fisher":>8} {"muSNR":>8}'
print('\n' + hdr)
print('-' * len(hdr))
for r in results:
    print(f'{r["subject"]:<5} {r["pair"]:<28} {r["condition"]:<6} {r["n_test"]:>4} '
          f'{r["train_acc"]:>9.3f} {r["acc"]:>7.3f} {r["amp"]:>8.3f} '
          f'{r["fisher"]:>8.3f} {r["mu_snr"]:>8.3f}')

---
## Figure 1 — Per-metric comparison (FES vs NOFES)

In [ ]:
METRICS = [
    ('acc',    'Classification accuracy',                            '0–1'),
    ('amp',    'Classification amplitude (mean |decision fn|)',      'a.u.'),
    ('fisher', 'Fisher ratio on LDA projection (test-set SNR)',      'a.u.'),
    ('mu_snr', 'μ-band power ratio  REST / MI  @ C3/Cz/C4',          'ratio'),
]

cond_color = {'FES': '#E05C2A', 'NOFES': '#2A7BE0'}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Online decoding: FES vs NOFES feedback  (per subject × offline-trained model)',
             fontsize=13, fontweight='bold', y=1.00)

for ax, (key, title, unit) in zip(axes.ravel(), METRICS):
    labels, vals, colors = [], [], []
    for subj in subjects:
        for pair in PAIRS:
            tag = pair['name'].split()[0]   # "Pair1" / "Pair2"
            for cond in ('FES', 'NOFES'):
                row = next((r for r in results
                            if r['subject']==subj and r['pair']==pair['name']
                            and r['condition']==cond), None)
                if row is None: continue
                labels.append(f'{subj}\n{tag}\n{cond}')
                vals.append(row[key])
                colors.append(cond_color[cond])
    x = np.arange(len(vals))
    ax.bar(x, vals, color=colors, edgecolor='white', zorder=2)
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=7.5)
    ax.set_title(f'{title}  ({unit})', fontsize=11, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    ax.spines[['top','right']].set_visible(False)
    if key == 'acc':
        ax.axhline(0.5, color='gray', linestyle='--', lw=0.8, alpha=0.6)

fig.legend(handles=[Patch(color=cond_color['FES'],   label='ONLINE_FES'),
                    Patch(color=cond_color['NOFES'], label='ONLINE_NOFES')],
           loc='upper right', ncol=2, bbox_to_anchor=(0.98, 1.0))
plt.tight_layout()
plt.savefig('fes_vs_nofes_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fes_vs_nofes_metrics.png')

---
## Figure 2 — LDA decision-function distributions

Visualizes classification amplitude and separability directly: wider FES vs NOFES spread between MI and REST curves = higher Fisher ratio and larger mean |margin|.

In [ ]:
fig, axes = plt.subplots(len(subjects), len(PAIRS),
                         figsize=(6 * len(PAIRS), 3.2 * len(subjects)),
                         sharex=True, squeeze=False)
fig.suptitle('LDA decision-function distributions on ONLINE sessions\n'
             '(class separation ↔ classification amplitude & SNR)',
             fontsize=12, fontweight='bold', y=1.01)

for i, subj in enumerate(subjects):
    for j, pair in enumerate(PAIRS):
        ax = axes[i][j]
        for cond, color in cond_color.items():
            row = next((r for r in results
                        if r['subject']==subj and r['pair']==pair['name']
                        and r['condition']==cond), None)
            if row is None: continue
            m_mi   = row['margin'][row['y_test'] == 1]
            m_rest = row['margin'][row['y_test'] == 0]
            ax.hist(m_mi,   bins=15, alpha=0.5,  color=color,
                    label=f'{cond} MI',   density=True)
            ax.hist(m_rest, bins=15, alpha=0.25, color=color, hatch='///',
                    edgecolor=color, label=f'{cond} REST', density=True)
        ax.axvline(0, color='k', lw=0.8)
        ax.set_title(f'{subj}  |  {pair["name"]}', fontsize=10, fontweight='bold')
        if j == 0:                     ax.set_ylabel('Density')
        if i == len(subjects) - 1:     ax.set_xlabel('LDA decision function')
        ax.spines[['top','right']].set_visible(False)
        if i == 0 and j == 0:
            ax.legend(fontsize=6.5, loc='upper left', ncol=2)

plt.tight_layout()
plt.savefig('decision_margin_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: decision_margin_distributions.png')

---
## Figure 3 — Paired Δ (FES − NOFES) per metric

Within each (subject × pair), FES and NOFES sessions use the same offline-trained model. Positive bars mean FES > NOFES; negative means NOFES > FES. This removes the offline-model-quality confound and isolates the effect of feedback type.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
fig.suptitle('Within-pair Δ = FES − NOFES  (positive → FES better)',
             fontsize=12, fontweight='bold', y=1.03)

for ax, (key, title, _) in zip(axes, METRICS):
    labels, deltas = [], []
    for subj in subjects:
        for pair in PAIRS:
            fes = next((r for r in results if r['subject']==subj and r['pair']==pair['name']
                        and r['condition']=='FES'),   None)
            nof = next((r for r in results if r['subject']==subj and r['pair']==pair['name']
                        and r['condition']=='NOFES'), None)
            if fes is None or nof is None: continue
            deltas.append(fes[key] - nof[key])
            labels.append(f'{subj}\n{pair["name"].split()[0]}')

    colors = ['#E05C2A' if d > 0 else '#2A7BE0' for d in deltas]
    ax.bar(np.arange(len(deltas)), deltas, color=colors, edgecolor='white', zorder=2)
    ax.axhline(0, color='k', lw=0.8)
    ax.set_xticks(np.arange(len(deltas)))
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_title(f'Δ {title.split("(")[0].strip()}', fontsize=10, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('fes_minus_nofes_delta.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fes_minus_nofes_delta.png')

---
## Summary Statistics

In [ ]:
agg = {c: {k: [] for k in ('acc','amp','fisher','mu_snr')} for c in ('FES','NOFES')}
for r in results:
    for k in agg[r['condition']]:
        agg[r['condition']][k].append(r[k])

n_pairs = len(agg['FES']['acc'])
print(f'=== Aggregate across {n_pairs} (subject × pair) comparisons ===\n')

hdr = f'{"Metric":<28} {"FES (mean ± sd)":>22} {"NOFES (mean ± sd)":>22} {"paired Δ":>12}'
print(hdr); print('-' * len(hdr))
for k, label in [('acc',    'Classification accuracy'),
                 ('amp',    'Classification amplitude'),
                 ('fisher', 'Fisher ratio (test SNR)'),
                 ('mu_snr', 'μ-band SNR  (REST/MI)')]:
    fes, nof = np.array(agg['FES'][k]), np.array(agg['NOFES'][k])
    delta    = fes - nof
    print(f'{label:<28} {fes.mean():>10.3f} ± {fes.std(ddof=1):>6.3f}  '
          f'{nof.mean():>10.3f} ± {nof.std(ddof=1):>6.3f}  {delta.mean():>+12.3f}')

# Sign test (simple)
print()
for k, label in [('acc','acc'), ('amp','|margin|'), ('fisher','Fisher'), ('mu_snr','μ-SNR')]:
    d = np.array(agg['FES'][k]) - np.array(agg['NOFES'][k])
    n_pos = int((d > 0).sum()); n_neg = int((d < 0).sum())
    print(f'  {label:<10}  FES > NOFES in {n_pos}/{len(d)} comparisons  (NOFES > FES in {n_neg})')